# GRACE Projection V2 — bắt đầu tại đây

Nhánh `projection-v2-local-metric` dùng residual MLP + local contrastive training. H1/ReVeal lịch sử đã lưu trên nhánh `reveal`; notebook này điều phối V2 và dùng cache chung.

| Bước | Chạy ở đâu | Cần chuyển môi trường khi nào |
|---|---|---|
| 1. Train và chọn head bằng validation retrieval proxy | Kaggle GPU, hoặc local | Chưa có head: cell tạo `projection-v2-training-inputs.zip` rồi dừng |
| 2. Retrieval/graph/prompt CPU | Colab CPU, hoặc local WSL | Chưa có frozen runs: cell tạo `projection-v2-cpu-inputs.zip` rồi dừng |
| 3. Preflight → LLM → evaluation | Local có Vertex ADC | Quay lại đây sau khi tải `projection-v2-heads.zip` và `projection-v2-cpu-prepared.zip` vào `bundles/` |

Mặc định chỉ chuẩn bị/đánh giá **validation ReVeal**, chưa gọi API mới. Đọc `docs/PROJECTION_V2_RESEARCH_AND_PLAN.md` và `docs/PROJECTION_V2_RUNBOOK.md` trước khi chuyển sang test.

In [ ]:
from pathlib import Path
import os,sys,json
candidates=([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h1').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H1_ROOT to the project folder')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
TRAIN_ON_KAGGLE=True
PREPARE_ON_COLAB=True  # False để prepare bằng Joern local WSL; dùng cùng graph cache.
RUN_INFERENCE=False   # True sẽ gọi Vertex cho prompt chưa có response cache.
from grace_h1.io import load_local_environment
from grace_h1.experiment_v2 import load_config,validate_config,require_heads,require_runs,train_all,prepare_all,preflight_all,infer_all,evaluate_all
from grace_h1.portable_v2 import create_bundle,import_bundle
load_local_environment(ROOT)
config=load_config(ROOT);validate_config(config)
HEADS=ROOT/'bundles/projection-v2-heads.zip'
PREPARED=ROOT/'bundles/projection-v2-cpu-prepared.zip'
print('Project:',ROOT,'\n',json.dumps(config,indent=2))


## Bước 1 — Kaggle train

Khi cell dưới dừng, upload zip được tạo lên Kaggle cùng `notebooks/kaggle_train_projection_v2.ipynb`. Bật GPU T4 và Internet, Run All. Tải `projection-v2-heads.zip` về `bundles/` rồi chạy lại từ đầu notebook này. Có thể tái sử dụng checkpoint V2 trong notebook Kaggle. Không cần CodeT5 weights; train dùng embedding cache.

In [ ]:
if HEADS.exists(): print(import_bundle(ROOT,HEADS,'heads'))
try:
    selection=require_heads(ROOT,config)
except (FileNotFoundError,KeyError,ValueError) as exc:
    if TRAIN_ON_KAGGLE:
        print(create_bundle(ROOT,ROOT/'bundles/projection-v2-training-inputs.zip','training_inputs'))
        raise RuntimeError('DỪNG BƯỚC 1: chạy notebooks/kaggle_train_projection_v2.ipynb trên Kaggle GPU; tải projection-v2-heads.zip về bundles/. '+str(exc)) from exc
    selection=train_all(ROOT,config)
for dataset,heads in selection['heads'].items():
    for objective,h in heads.items():
        print(dataset,objective,h['selection_status'],'best epoch',h['best']['epoch'],'macro P@5',h['best']['retrieval']['macro_label_precision_at_5'])


## Bước 2 — CPU preparation

Nếu head chỉ có `baseline_fallback`, notebook dừng trước khi tạo graph: chưa có proxy gain để chạy LLM. Đọc history và điều chỉnh bằng validation. `allow_identity_fallback_run=true` chỉ dùng khi chủ động đo identity control.

Chạy Colab: đặt `projection-v2-cpu-inputs.zip`, `projection-v2-heads.zip`, và Joern runtime có sẵn `grace-h1-joern-runtime.zip` vào **MyDrive/GRACE-projection-v2-transfer**. Chạy `notebooks/colab_cpu_prepare_v2.ipynb` với accelerator **None**. Tải `projection-v2-cpu-prepared.zip` về `bundles/` rồi quay lại đây. Muốn chạy CPU local WSL, đổi `PREPARE_ON_COLAB=False` và đặt `JOERN_HOME` hợp lệ.

In [ ]:
blocked=[f'{d}/{a}' for d in config['datasets'] for a in config['llm_arms'] if a!='baseline_whitening' and selection['heads'][d][a.removeprefix('h1_')]['selection_status']=='baseline_fallback']
if blocked and not config.get('allow_identity_fallback_run',False):
    raise RuntimeError('Chưa có validation retrieval proxy gain: '+', '.join(blocked)+'. Tiếp tục nghiên cứu offline trước khi chạy LLM.')
if PREPARED.exists(): print(import_bundle(ROOT,PREPARED,'prepared'))
try:
    run_rows=require_runs(ROOT,config)
    print('Dùng lại frozen V2 preparation đã có.')
except (FileNotFoundError,KeyError,ValueError) as exc:
    if PREPARE_ON_COLAB:
        print(create_bundle(ROOT,ROOT/'bundles/projection-v2-cpu-inputs.zip','cpu_inputs'))
        raise RuntimeError('DỪNG BƯỚC 2: chạy notebooks/colab_cpu_prepare_v2.ipynb; tải projection-v2-cpu-prepared.zip về bundles/. '+str(exc)) from exc
    run_rows=prepare_all(ROOT,config)
print(run_rows)


## Bước 3 — Preflight và inference local

Cell preflight không gọi API. Kiểm tra số request thiếu, rồi đặt `RUN_INFERENCE=True` để chạy Vertex bằng ADC local. Hai arm dùng cùng prompt template/graph/query cohort/generation settings. Response cache chỉ được tái sử dụng khi exact prompt và settings khớp.

In [ ]:
plans=preflight_all(ROOT,config)
for plan in plans: print(plan)


In [ ]:
if RUN_INFERENCE:
    inference=infer_all(ROOT,config,allow_api=True)
    for item in inference: print(item)
else:
    print('Chưa gọi API. Đặt RUN_INFERENCE=True sau khi xem preflight nếu muốn chạy inference.')


## Bước 4 — Paired evaluation và chuyển sang test

Báo cáo V2 nằm trong `runs/projection_v2/experiment_summary.json`; bảng retrieval proxy không thay thế F1/MCC của GRACE. Chỉ sau khi đánh giá validation, lưu cấu hình và checkpoint đã chọn, rồi đổi `splits` thành `["test"]` và `test_protocol_frozen=true` trong `configs/projection_v2.json`. Đây là xác nhận protocol của người chạy, không phải bằng chứng đã chạy validation. Training identity không đổi khi đổi split; Bước 1 dùng lại head, Bước 2 tạo input CPU mới. Không dùng test để chọn hyperparameter/seed.

In [ ]:
if all((ROOT/r['path']/'predictions.jsonl').exists() for r in run_rows):
    report=evaluate_all(ROOT,config)
    print('Status:',report['status'])
    for row in report['scores']: print(row['dataset'],row['arm'],row['metrics'])
    for comparison in report['comparisons']: print(comparison)
else:
    print('Chưa có đủ predictions; tiếp tục Bước 3 trước khi evaluation.')
